# Doctor Scheduling - Basic Approach

This notebook explores a simple and practical approach to implementing
doctor scheduling.


### Domain Models

In [1]:
public record Patient(
    int Id, 
    string Name
);

public record Doctor(
    int Id, 
    string Name
);

public record DoctorSchedule(
    int DoctorId,
    DayOfWeek Day,
    TimeOnly Start,
    TimeOnly End,
    short SlotDuration // minutes
);

public record DoctorException(
    int DoctorId,
    DateOnly Date,
    TimeOnly? OffStart,
    TimeOnly? OffEnd,
    bool IsFullDayOff
);

public record Appointment(
    int DoctorId,
    int PatientId,
    DateOnly Date,
    TimeOnly Start,
    TimeOnly End,
    AppointmentStatus Status
);

public enum AppointmentStatus{ Booked }

### Get Available Slots

In [2]:
public record AvailableDateWithSlots(
    DateOnly Date, 
    List<Slot> Slots
);

public record Slot(
    TimeOnly Start, 
    TimeOnly End,
    bool IsBooked
);

In [ ]:
public List<AvailableDateWithSlots> GetAvailableDatesWithSlots(
    int doctorId,
    DateOnly from,
    DateOnly to,
    List<DoctorSchedule> schedules,
    List<DoctorException> exceptions,
    List<Appointment> appointments)
{
    var result = new List<AvailableDateWithSlots>();

    for (DateOnly date = from; date <= to; date = date.AddDays(1))
    {
        var schedule = schedules.FirstOrDefault(s => s.DoctorId == doctorId && s.Day == date.DayOfWeek);
        if (schedule is null)
            continue;

        var exception = exceptions.FirstOrDefault(e => e.DoctorId == doctorId && e.Date == date);
        if (exception?.IsFullDayOff == true)
            continue;

        var slots = new List<Slot>();

        for (TimeOnly time = schedule.Start;
             time.AddMinutes(schedule.SlotDuration) <= schedule.End; 
             time = time.AddMinutes(schedule.SlotDuration))
        {
            TimeOnly end = time.AddMinutes(schedule.SlotDuration);

            if (exception?.IsFullDayOff == false && 
                time < exception.OffEnd && 
                end > exception.OffStart)
                continue;
            
            var isBooked = appointments
                .Any(a => 
                    a.DoctorId == doctorId &&
                    a.Date == date &&
                    a.Start == time &&
                    a.Status == AppointmentStatus.Booked);
            
            slots.Add(new Slot(time, end, isBooked));
        }

        result.Add(new AvailableDateWithSlots(date, slots));
    }

    return result;
}

### Example

In [4]:
public static class DateHelpers
{
	public static DateOnly Next(DateOnly from, DayOfWeek dayOfWeek)
	{
		int diff = ((int)dayOfWeek - (int)from.DayOfWeek + 7) % 7;
		if (diff == 0) diff = 7; // next occurrence
		return from.AddDays(diff);
	}
}

In [ ]:
DateOnly saturday = DateHelpers.Next(
    DateOnly.FromDateTime(DateTime.Today), 
    DayOfWeek.Saturday);

var patient = new Patient(1, "Patient Name");

var doctor = new Doctor(1, "Dr.Test");

var schedules = new List<DoctorSchedule>
{
    new DoctorSchedule(1, saturday.DayOfWeek, new TimeOnly(09, 00), new TimeOnly(12, 00), 15),
    new DoctorSchedule(1, saturday.AddDays(1).DayOfWeek, new TimeOnly(09, 00), new TimeOnly(12, 00), 15),
    new DoctorSchedule(1, saturday.AddDays(2).DayOfWeek, new TimeOnly(08, 00), new TimeOnly(10, 00), 15)
};

var exceptions = new List<DoctorException>
{
    new DoctorException(1, saturday.AddDays(1), new TimeOnly(09, 00), new TimeOnly(10, 00), false),
    new DoctorException(1, saturday.AddDays(2), null, null, true)
};

var appointments = new List<Appointment>
{
    new Appointment(1, 1, saturday, new TimeOnly(09, 15), new TimeOnly(09, 30), AppointmentStatus.Booked),
    new Appointment(1, 1, saturday, new TimeOnly(09, 30), new TimeOnly(09, 45), AppointmentStatus.Booked),
};

var from = DateOnly.FromDateTime(DateTime.Today);
var to = from.AddDays(7);

var availableDatesWithSlots = GetAvailableDatesWithSlots(1, from, to, schedules, exceptions, appointments);

availableDatesWithSlots

index value 0 AvailableDateWithSlots { Date = 1/10/2026, Slots = System.Collections.Generic.List`1[Submission#2+Slot] } Date 1/10/2026 Year 2026 Month 1 Day 10 DayOfWeek Saturday DayOfYear 10 DayNumber 739625 Slots index value 0 Slot { Start = 9:00 AM, End = 9:15 AM, IsBooked = False } Start 9:00 AM Hour 9 Minute 0 Second 0 Millisecond 0 Microsecond 0 Nanosecond 0 Ticks 324000000000 End 9:15 AM Hour 9 Minute 15 Second 0 Millisecond 0 Microsecond 0 Nanosecond 0 Ticks 333000000000 IsBooked False 1 Slot { Start = 9:15 AM, End = 9:30 AM, IsBooked = True } Start 9:15 AM Hour 9 Minute 15 Second 0 Millisecond 0 Microsecond 0 Nanosecond 0 Ticks 333000000000 End 9:30 AM Hour 9 Minute 30 Second 0 Millisecond 0 Microsecond 0 Nanosecond 0 Ticks 342000000000 IsBooked True 2 Slot { Start = 9:30 AM, End = 9:45 AM, IsBooked = True } Start 9:30 AM Hour 9 Minute 30 Second 0 Millisecond 0 Microsecond 0 Nanosecond 0 Ticks 342000000000 End 9:45 AM Hour 9 Minute 45 Second 0 Millisecond 0 Microsecond 0 Nanosecond 0 Ticks 351000000000 IsBooked True 3 Slot { Start = 9:45 AM, End = 10:00 AM, IsBooked = False } Start 9:45 AM Hour 9 Minute 45 Second 0 Millisecond 0 Microsecond 0 Nanosecond 0 Ticks 351000000000 End 10:00 AM Hour 10 Minute 0 Second 0 Millisecond 0 Microsecond 0 Nanosecond 0 Ticks 360000000000 IsBooked False 4 Slot { Start = 10:00 AM, End = 10:15 AM, IsBooked = False } Start 10:00 AM Hour 10 Minute 0 Second 0 Millisecond 0 Microsecond 0 Nanosecond 0 Ticks 360000000000 End 10:15 AM Hour 10 Minute 15 Second 0 Millisecond 0 Microsecond 0 Nanosecond 0 Ticks 369000000000 IsBooked False 5 Slot { Start = 10:15 AM, End = 10:30 AM, IsBooked = False } Start 10:15 AM Hour 10 Minute 15 Second 0 Millisecond 0 Microsecond 0 Nanosecond 0 Ticks 369000000000 End 10:30 AM Hour 10 Minute 30 Second 0 Millisecond 0 Microsecond 0 Nanosecond 0 Ticks 378000000000 IsBooked False 6 Slot { Start = 10:30 AM, End = 10:45 AM, IsBooked = False } Start 10:30 AM Hour 10 Minute 30 Second 0 Millisecond 0 Microsecond 0 Nanosecond 0 Ticks 378000000000 End 10:45 AM Hour 10 Minute 45 Second 0 Millisecond 0 Microsecond 0 Nanosecond 0 Ticks 387000000000 IsBooked False 7 Slot { Start = 10:45 AM, End = 11:00 AM, IsBooked = False } Start 10:45 AM Hour 10 Minute 45 Second 0 Millisecond 0 Microsecond 0 Nanosecond 0 Ticks 387000000000 End 11:00 AM Hour 11 Minute 0 Second 0 Millisecond 0 Microsecond 0 Nanosecond 0 Ticks 396000000000 IsBooked False 8 Slot { Start = 11:00 AM, End = 11:15 AM, IsBooked = False } Start 11:00 AM Hour 11 Minute 0 Second 0 Millisecond 0 Microsecond 0 Nanosecond 0 Ticks 396000000000 End 11:15 AM Hour 11 Minute 15 Second 0 Millisecond 0 Microsecond 0 Nanosecond 0 Ticks 405000000000 IsBooked False 9 Slot { Start = 11:15 AM, End = 11:30 AM, IsBooked = False } Start 11:15 AM Hour 11 Minute 15 Second 0 Millisecond 0 Microsecond 0 Nanosecond 0 Ticks 405000000000 End 11:30 AM Hour 11 Minute 30 Second 0 Millisecond 0 Microsecond 0 Nanosecond 0 Ticks 414000000000 IsBooked False 10 Slot { Start = 11:30 AM, End = 11:45 AM, IsBooked = False } Start 11:30 AM Hour 11 Minute 30 Second 0 Millisecond 0 Microsecond 0 Nanosecond 0 Ticks 414000000000 End 11:45 AM Hour 11 Minute 45 Second 0 Millisecond 0 Microsecond 0 Nanosecond 0 Ticks 423000000000 IsBooked False 11 Slot { Start = 11:45 AM, End = 12:00 PM, IsBooked = False } Start 11:45 AM Hour 11 Minute 45 Second 0 Millisecond 0 Microsecond 0 Nanosecond 0 Ticks 423000000000 End 12:00 PM Hour 12 Minute 0 Second 0 Millisecond 0 Microsecond 0 Nanosecond 0 Ticks 432000000000 IsBooked False 1 AvailableDateWithSlots { Date = 1/11/2026, Slots = System.Collections.Generic.List`1[Submission#2+Slot] } Date 1/11/2026 Year 2026 Month 1 Day 11 DayOfWeek Sunday DayOfYear 11 DayNumber 739626 Slots index value 0 Slot { Start = 10:00 AM, End = 10:15 AM, IsBooked = False } Start 10:00 AM Hour 10 Minute 0 Second 0 Millisecond 0 Microsecond 0 Nanosecond 0 Ticks 360000000000 End 10:15 AM Hour 10 Minute 15 Second 0 Millisecond 0 Microsecond 0 Nanosecond